# Online data

In [1]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import random
from pathlib import Path
from collections import defaultdict

# --- Configuration ---
#from the root of the project directory, adjust if running from a different location:
ROOT_DIR = Path("../../")
DATA_DIR = Path(ROOT_DIR / "data/processed/online")
SPLITS = ['train', 'valid', 'test']
OUTPUT_PLOT_DIR = Path(ROOT_DIR / "experiments/figures")
OUTPUT_PLOT_DIR.mkdir(parents=True, exist_ok=True)

# For faster plotting, sample at most this many points per split
MAX_POINTS_FOR_PLOT = 100000

In [5]:
# --- Load Data ---
all_stats = {}
print("Loading and analyzing data...")

for split in SPLITS:
    file_path = DATA_DIR / f"{split}.pt"
    if not file_path.exists():
        print(f"Warning: {file_path} not found, skipping.")
        continue
    
    data = torch.load(file_path, weights_only=False)  # Load as list of dicts
    print(f"Loaded {len(data)} samples from {split} split.")
    
    # Accumulators
    total_samples = len(data)
    total_points = 0
    total_strokes = 0
    stroke_counts = []
    seq_lengths = []
    
    # For spatial distribution (sampling)
    sampled_x = []
    sampled_y = []
    sampled_t = []
    
    # Determine sampling rate
    # First pass: estimate total points
    est_total_points = sum(sum(s.shape[0] for s in sample['strokes']) for sample in data)
    sample_rate = min(1.0, MAX_POINTS_FOR_PLOT / max(est_total_points, 1))
    
    for sample in data:
        strokes = sample['strokes']
        n_strokes = len(strokes)
        n_points = sum(stroke.shape[0] for stroke in strokes)
        
        stroke_counts.append(n_strokes)
        seq_lengths.append(n_points)
        total_strokes += n_strokes
        total_points += n_points
        
        # Sample points for plots
        for stroke in strokes:
            for i in range(stroke.shape[0]):
                if random.random() < sample_rate:
                    sampled_x.append(stroke[i, 0].item())
                    sampled_y.append(stroke[i, 1].item())
                    sampled_t.append(stroke[i, 2].item())
    
    # Compute statistics
    stats = {
        'total_samples': total_samples,
        'total_points': total_points,
        'total_strokes': total_strokes,
        'avg_points_per_sample': total_points / total_samples,
        'avg_strokes_per_sample': total_strokes / total_samples,
        'min_seq_len': min(seq_lengths),
        'max_seq_len': max(seq_lengths),
        'median_seq_len': np.median(seq_lengths),
        'std_seq_len': np.std(seq_lengths),
        'min_strokes': min(stroke_counts),
        'max_strokes': max(stroke_counts),
        'median_strokes': np.median(stroke_counts),
        'std_strokes': np.std(stroke_counts),
        'seq_lengths': seq_lengths,
        'stroke_counts': stroke_counts,
        'sampled_x': sampled_x,
        'sampled_y': sampled_y,
        'sampled_t': sampled_t
    }
    all_stats[split] = stats

    # Print statistics
    print(f"\n--- {split.upper()} SPLIT STATISTICS ---")
    print(f"Samples: {stats['total_samples']}")
    print(f"Total points: {stats['total_points']:,}")
    print(f"Average points per sample: {stats['avg_points_per_sample']:.2f}")
    print(f"Sequence length - Min: {stats['min_seq_len']}, Max: {stats['max_seq_len']}, Median: {stats['median_seq_len']:.0f}, Std: {stats['std_seq_len']:.1f}")
    print(f"Average strokes per sample: {stats['avg_strokes_per_sample']:.2f}")
    print(f"Strokes per sample - Min: {stats['min_strokes']}, Max: {stats['max_strokes']}, Median: {stats['median_strokes']:.0f}, Std: {stats['std_strokes']:.1f}")
    print(f"Sampled {len(sampled_x)} points for plots (sample_rate = {sample_rate:.3f})")

# --- Generate Plots ---
if 'train' in all_stats:
    train_stats = all_stats['train']
    
    # Plot 1: Histogram of sequence lengths
    plt.figure(figsize=(10, 6))
    plt.hist(train_stats['seq_lengths'], bins=50, alpha=0.7, edgecolor='black', color='steelblue')
    plt.title('Distribution of Sequence Lengths (Total Points per Sample) - Training Set')
    plt.xlabel('Sequence Length (number of points)')
    plt.ylabel('Frequency')
    plt.axvline(train_stats['median_seq_len'], color='red', linestyle='dashed', linewidth=2, label=f"Median: {train_stats['median_seq_len']:.0f}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(OUTPUT_PLOT_DIR / 'seq_length_distribution.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved sequence length plot to {OUTPUT_PLOT_DIR / 'seq_length_distribution.png'}")

    # Plot 2: Histogram of stroke counts
    plt.figure(figsize=(10, 6))
    plt.hist(train_stats['stroke_counts'], bins=50, alpha=0.7, edgecolor='black', color='seagreen')
    plt.title('Distribution of Strokes per Sample - Training Set')
    plt.xlabel('Number of Strokes')
    plt.ylabel('Frequency')
    plt.axvline(train_stats['median_strokes'], color='red', linestyle='dashed', linewidth=2, label=f"Median: {train_stats['median_strokes']:.0f}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(OUTPUT_PLOT_DIR / 'stroke_count_distribution.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved stroke count plot to {OUTPUT_PLOT_DIR / 'stroke_count_distribution.png'}")

    # Plot 3: Spatial distribution (using sampled points - much faster)
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].hist(train_stats['sampled_x'], bins=100, alpha=0.7, color='blue', edgecolor='black')
    axes[0].set_title('Normalized X Coordinates')
    axes[0].set_xlabel('X')
    axes[0].set_ylabel('Frequency')
    axes[0].grid(True, alpha=0.3)
    
    axes[1].hist(train_stats['sampled_y'], bins=100, alpha=0.7, color='green', edgecolor='black')
    axes[1].set_title('Normalized Y Coordinates')
    axes[1].set_xlabel('Y')
    axes[1].set_ylabel('Frequency')
    axes[1].grid(True, alpha=0.3)
    
    axes[2].hist(train_stats['sampled_t'], bins=100, alpha=0.7, color='purple', edgecolor='black')
    axes[2].set_title('Normalized Timestamps (per stroke)')
    axes[2].set_xlabel('Normalized Time')
    axes[2].set_ylabel('Frequency')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_PLOT_DIR / 'spatial_temporal_distribution.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved spatial/temporal distribution plot to {OUTPUT_PLOT_DIR / 'spatial_temporal_distribution.png'}")

    # Plot 4: Comparison across splits (box plot)
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
    # Box plot for sequence lengths
    seq_data = [all_stats[split]['seq_lengths'] for split in SPLITS if split in all_stats]
    axes[0].boxplot(seq_data, labels=[s for s in SPLITS if s in all_stats])
    axes[0].set_title('Sequence Length Distribution Across Splits')
    axes[0].set_ylabel('Sequence Length (points)')
    axes[0].grid(True, alpha=0.3)
    
    # Box plot for stroke counts
    stroke_data = [all_stats[split]['stroke_counts'] for split in SPLITS if split in all_stats]
    axes[1].boxplot(stroke_data, labels=[s for s in SPLITS if s in all_stats])
    axes[1].set_title('Stroke Count Distribution Across Splits')
    axes[1].set_ylabel('Number of Strokes')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_PLOT_DIR / 'split_comparison.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved split comparison plot to {OUTPUT_PLOT_DIR / 'split_comparison.png'}")

print("\nData analysis complete!")

Loading and analyzing data...
Loaded 16717 samples from train split.

--- TRAIN SPLIT STATISTICS ---
Samples: 16717
Total points: 40,589,216
Average points per sample: 2428.02
Sequence length - Min: 2, Max: 7268, Median: 2288, Std: 976.2
Average strokes per sample: 35.29
Strokes per sample - Min: 1, Max: 161, Median: 30, Std: 20.9
Sampled 100168 points for plots (sample_rate = 0.002)
Loaded 2785 samples from valid split.

--- VALID SPLIT STATISTICS ---
Samples: 2785
Total points: 6,958,479
Average points per sample: 2498.56
Sequence length - Min: 3, Max: 5824, Median: 2372, Std: 897.4
Average strokes per sample: 35.96
Strokes per sample - Min: 1, Max: 150, Median: 32, Std: 19.3
Sampled 100363 points for plots (sample_rate = 0.014)
Loaded 2785 samples from test split.

--- TEST SPLIT STATISTICS ---
Samples: 2785
Total points: 6,696,596
Average points per sample: 2404.52
Sequence length - Min: 2, Max: 6525, Median: 2223, Std: 1196.7
Average strokes per sample: 43.03
Strokes per sample - 

C:\Users\Korisnik\AppData\Local\Temp\ipykernel_22208\3320896773.py:143: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  axes[0].boxplot(seq_data, labels=[s for s in SPLITS if s in all_stats])
C:\Users\Korisnik\AppData\Local\Temp\ipykernel_22208\3320896773.py:150: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  axes[1].boxplot(stroke_data, labels=[s for s in SPLITS if s in all_stats])


Saved split comparison plot to ..\..\experiments\figures\split_comparison.png

Data analysis complete!


In [9]:
# --- Generate LaTeX Table ---
print("\n" + "="*50)
print("LATEX TABLE FOR YOUR REPORT")
print("="*50)

print("\n\\begin{table}[h]")
print("\\centering")
print("\\caption{Online Handwriting Data Statistics (DIDI Dataset)}")
print("\\begin{tabular}{l|c|c|c}")
print("\\toprule")
print("Statistic & Training & Validation & Test \\\\")
print("\\midrule")

# Get stats
train = all_stats['train']
valid = all_stats['valid']
test = all_stats['test']

print(f"Samples & {train['total_samples']:,} & {valid['total_samples']:,} & {test['total_samples']:,} \\\\")
print(f"Total points & {train['total_points']:,} & {valid['total_points']:,} & {test['total_points']:,} \\\\")
print(f"Avg. points/sample & {train['avg_points_per_sample']:.1f} & {valid['avg_points_per_sample']:.1f} & {test['avg_points_per_sample']:.1f} \\\\")
print(f"Median points/sample & {train['median_seq_len']:.0f} & {valid['median_seq_len']:.0f} & {test['median_seq_len']:.0f} \\\\")
print(f"Min/Max points & {train['min_seq_len']}/{train['max_seq_len']} & {valid['min_seq_len']}/{valid['max_seq_len']} & {test['min_seq_len']}/{test['max_seq_len']} \\\\")
print(f"Avg. strokes/sample & {train['avg_strokes_per_sample']:.1f} & {valid['avg_strokes_per_sample']:.1f} & {test['avg_strokes_per_sample']:.1f} \\\\")
print(f"Median strokes/sample & {train['median_strokes']:.0f} & {valid['median_strokes']:.0f} & {test['median_strokes']:.0f} \\\\")
print(f"Min/Max strokes & {train['min_strokes']}/{train['max_strokes']} & {valid['min_strokes']}/{valid['max_strokes']} & {test['min_strokes']}/{test['max_strokes']} \\\\")

print("\\bottomrule")
print("\\end{tabular}")
print("\\label{tab:online_data_stats}")
print("\\end{table}")


LATEX TABLE FOR YOUR REPORT

\begin{table}[h]
\centering
\caption{Online Handwriting Data Statistics (DIDI Dataset)}
\begin{tabular}{l|c|c|c}
\toprule
Statistic & Training & Validation & Test \\
\midrule
Samples & 16,717 & 2,785 & 2,785 \\
Total points & 40,589,216 & 6,958,479 & 6,696,596 \\
Avg. points/sample & 2428.0 & 2498.6 & 2404.5 \\
Median points/sample & 2288 & 2372 & 2223 \\
Min/Max points & 2/7268 & 3/5824 & 2/6525 \\
Avg. strokes/sample & 35.3 & 36.0 & 43.0 \\
Median strokes/sample & 30 & 32 & 39 \\
Min/Max strokes & 1/161 & 1/150 & 1/147 \\
\bottomrule
\end{tabular}
\label{tab:online_data_stats}
\end{table}


# Offline data